<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
!از صورتش معلومه
</font>
</h1>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مقدمه و صورت مسئله
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
هدف این مسئله تشخیص جنسیت و تخمین گروه سنی افراد از روی تصاویر چهره‌ی آن‌ها است. برای انجام این کار، مجموعه‌ای از تصاویر چهره‌ی افراد به همراه ویژگی‌های مربوط به هر چهره در دسترس است. با استفاده از این مجموعه‌داده، قصد داریم مدلی بسازیم که بتواند با تحلیل تصاویر چهره، جنسیت و گروه سنی افراد را به درستی تشخیص دهد و پیش‌بینی کند.
</font>
</p>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
وارد کردن کتابخانه‌های مورد نیاز
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    ابتدا کتابخانه‌های مورد نیازتان را وارد کنید.
</font>
</p>

In [1]:
!pip install gdown

In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import gdown
import zipfile

I0000 00:00:1790238681.012022  529491 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790238682.190290  529491 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790238691.790608  529491 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


<h2 align="right" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
معرفی مجموعه داده
</font>
</h2>

<p dir="rtl" style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
مجموعه داده‌های تصویری شامل ۴۴۷۷۵ تصویر می‌باشد که شامل داده‌های آموزش و نمونه‌های آزمون است. تصاویر مربوط به هر شخص درون دایرکتوری خود قرار دارند. از هر شخص تعداد متغیری از تصاویر موجود است.
</font>
</p>

<p dir="rtl" style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
ویژگی‌های تصاویر مربوط به داده‌های آموزش در یک فایل برچسب‌گذاری با نام <code>train.csv</code> قرار دارد که شامل ۳۵۳۳۲ سطر است.
در این فایل ۴۷ ویژگی وجود دارد. داشتن یک ویژگی با عدد ۱+ نشان داده می‌شود، نداشتن یک ویژگی با عدد ۱- نشان داده می‌شود و نامعلوم بودن داشتن یا نداشتن یک ویژگی با عدد ۰ مشخص می‌شود.
توضیحات مربوط به ستون های مهم در این مسئله در جدول زیر آمده است.
</font>
</p>


<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|Filename|اسم و دایرکتوری مربوط به تصویر|
|Identity|شماره‌ای یکتای مربوط به شخص درون تصویر|
|Male|مرد بودن|
|Young|جوان بودن|
|Middle_Aged|میانسال بودن|
|Senior|پیر بودن|
    
</font>
</div>
</center>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font color="red"><b color='red'>توجه:</b></font>
<font face="vazir" size=3>
دقت کنید که قرار گرفتن ۱+ در ستون <code>Male</code> به معنی مرد بودن و ۱- در این ستون به معنی زن بودن است.
</font>
</p>



<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
خواندن مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    در ابتدا نیاز است فایل‌های مجموعه‌داده را بخوانید. ویژگی‌های آموزشی در فایل <code>train.csv</code> و دایرکتوری مربوط به تصاویر نمونه‌های آزمون که باید دسته‌ی آن‌ها را پیش‌بینی کنید در فایل <code>test.csv</code> ذخیره شده‌اند. اگر لازم دانستید می‌توانید به دلخواه خود بخشی از دادگان آموزشی را به عنوان دادگان اعتبارسنجی نیز جدا کنید. دقت کنید که ستون های هدف از داده های آزمون حذف شده‌اند و هدف وجود این فایل دسترسی به دایرکتوری مربوط به تصاویر داده‌‌های آزمون است.
</font>
</p>


<p dir="rtl" style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
    <font face="vazir">
        <span style="color:orange"><b>نکته:</b></span>
        حجم فایل آموزشی و آزمون به‌ترتیب حدود ۶۵۵ مگابایت است. در صورتی‌که بارگذاری این فایل‌ها در محیط کولب برای‌تان دشوار است می‌توانید با استفاده از دستور زیر این دو فایل را به‌صورت مستقیم در محیط کولب بارگیری کنید.
        همچنین می‌توانید از لینک درایو زیر فایل‌ها را دانلود کرده و بر روی محیط دستگاه خود از آن استفاده کنید.
    </font>
</p>

<p dir="rtl" style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
    <font face="vazir">
        <a href="https://drive.google.com/file/d/1_mGrKY786DV8OpT9Xw5gmxJcY0aq3Bb8/view?usp=share_link" target="_blank" style="color:blue;">لینک دانلود از گوگل درایو</a>
    </font>
</p>





In [4]:
destination = 'downloaded_file.zip'
gdown.download(f'https://drive.google.com/file/d/1qMsyk9sMrdzqr7lzliIhJPJy01HbRa9I/view', destination, quiet=False)

with zipfile.ZipFile(destination, 'r') as zip_ref:
    zip_ref.extractall('data')

print("Download and extraction complete.")

 13%|█▎        | 71.3M/560M [00:39<02:22, 3.44MB/s]Downloading...
From (original): https://drive.google.com/uc?id=1qMsyk9sMrdzqr7lzliIhJPJy01HbRa9I
From (redirected): https://drive.google.com/uc?id=1qMsyk9sMrdzqr7lzliIhJPJy01HbRa9I&confirm=t&uuid=06127d6b-55c9-4da1-9d8e-b25256b756da
To: /home/norbert/Projects/computer_vision_oly/classification/her_face/downloaded_file.zip


KeyboardInterrupt: 

In [3]:
train_data = pd.read_csv('data/train.csv') 
train_data.head()

,Filename,Identity,Male,Young,Middle_Aged,Senior,Asian,White,Black,Rosy_Cheeks,...,Big_Nose,Pointy_Nose,Heavy_Makeup,Wearing_Hat,Wearing_Earrings,Wearing_Necktie,Wearing_Lipstick,No_Eyewear,Eyeglasses,Attractive
0,n000003/0003_01.jpg,3,1,0,0,-1,-1,1,-1,-1,...,0,0,-1,-1,-1,0,-1,1,-1,-1
1,n000003/0009_02.jpg,3,1,-1,-1,1,-1,1,-1,-1,...,0,0,-1,-1,-1,0,-1,1,-1,-1
2,n000003/0010_01.jpg,3,1,0,0,-1,-1,1,-1,-1,...,0,0,-1,-1,-1,0,-1,1,-1,-1
3,n000003/0012_01.jpg,3,1,-1,-1,-1,-1,1,-1,-1,...,0,0,-1,-1,-1,0,-1,1,-1,-1
4,n000003/0014_01.jpg,3,1,0,0,-1,-1,1,-1,-1,...,0,0,-1,-1,-1,0,-1,1,-1,-1


In [5]:
test_data = pd.read_csv('data/test.csv')
test_data.head()

,Filename,Identity,Asian,White,Black,Rosy_Cheeks,Shiny_Skin,Bald,Wavy_Hair,Receding_Hairline,...,Big_Nose,Pointy_Nose,Heavy_Makeup,Wearing_Hat,Wearing_Earrings,Wearing_Necktie,Wearing_Lipstick,No_Eyewear,Eyeglasses,Attractive
0,n000002/0001_01.jpg,2,-1,1,-1,0,0,-1,1,-1,...,-1,1,1,-1,1,-1,1,1,-1,1
1,n000002/0012_01.jpg,2,-1,1,-1,0,0,-1,1,-1,...,-1,1,1,-1,1,-1,1,1,-1,1
2,n000002/0013_01.jpg,2,-1,1,-1,0,0,-1,1,-1,...,-1,1,1,-1,1,-1,1,1,-1,1
3,n000002/0018_04.jpg,2,-1,1,-1,-1,0,-1,1,-1,...,-1,1,1,-1,1,-1,1,1,-1,1
4,n000002/0020_01.jpg,2,-1,1,-1,0,1,-1,1,-1,...,-1,1,1,-1,1,-1,1,1,-1,1



<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
سپس نیاز است که تصاویر را از دایرکتوری های قرار گرفته در فایل های <code>test.csv</code> و <code>train.csv</code> بخوانید.
</font>
</p>

In [7]:
image_base_dir = 'data/image_data'
train_data['image_path'] = train_data['Filename'].apply(lambda x: os.path.join(image_base_dir, str(x)))
test_data['image_path'] = test_data['Filename'].apply(lambda x: os.path.join(image_base_dir, str(x)))

print("train_data : ", len(train_data))
print("test_data : ", len(test_data))

train_data :  35332
test_data :  8702


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
پیش‌پردازش و مهندسی ویژگی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    در این سوال شما می‌توانید از هر تکنیک پیش‌پردازش/مهندسی ویژگی که در گذشته آموختید، استفاده کنید.
    <br>
    تکنیک‌هایی که استفاده می‌کنید به شکل مستقیم مورد ارزیابی توسط سامانه داوری قرار <b>نمی‌گیرند.</b> بلکه همه آن‌ها در دقت مدل شما تاثیر خواهند گذاشت؛ بنابراین هر چه پیش‌پردازش/مهندسی ویژگی بهتری انجام دهید تا دقت مدل بهبود پیدا کند، امتیاز بیشتری از این سوال کسب خواهید کرد.
</font>
</p>

In [8]:
target_cols = ['Male', 'Young', 'Middle_Aged', 'Senior']
mapping = {1: 1.0, -1: 0.0, 0: 0.5}

for col in target_cols:
    train_data[col] = train_data[col].map(mapping)

train_df, val_df = train_test_split(train_data, test_size=0.1, random_state=42)

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='image_path',
    y_col=target_cols,
    target_size=(128, 128),
    batch_size=32,
    class_mode='raw'
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='image_path',
    y_col=target_cols,
    target_size=(128, 128),
    batch_size=32,
    class_mode='raw',
    shuffle=False
)

Found 31798 validated image filenames.
Found 3534 validated image filenames.


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مدل‌سازی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    حال که داده را پاکسازی کرده و احتمالا ویژگی‌هایی را به آن افزوده یا از آن حذف کرده‌اید، وقت آن است که مدلی آموزش دهید که بتواند متغیر هدف این مسئله را پیش‌بینی کند.

</font>
</p>

In [9]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model.trainable = False 

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x) 
x = Dense(128, activation='relu')(x)

predictions = Dense(4, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

I0000 00:00:1789812959.508642 3064401 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4035 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Epoch 1/5


I0000 00:00:1789812961.039556 3064401 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1789812963.477504 3068890 service.cc:153] XLA service 0x788c54032f80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789812963.477531 3068890 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1789812963.563469 3068890 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789812964.193513 3068890 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1789812964.257353 3068890 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8148__.117
E0000 00:00:1789812967.144399 3068890 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup e

  5/994 ━━━━━━━━━━━━━━━━━━━━ 29s 30ms/step - accuracy: 0.5047 - loss: 0.7737 

I0000 00:00:1789812973.859456 3068890 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


620/994 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - accuracy: 0.8581 - loss: 0.3615

I0000 00:00:1789812993.401798 3068891 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8148__.117
E0000 00:00:1789812996.220920 3068891 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


993/994 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.8654 - loss: 0.3481

E0000 00:00:1789813023.756311 3068892 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


994/994 ━━━━━━━━━━━━━━━━━━━━ 66s 54ms/step - accuracy: 0.8801 - loss: 0.3223 - val_accuracy: 0.9004 - val_loss: 0.2995
Epoch 2/5
994/994 ━━━━━━━━━━━━━━━━━━━━ 36s 36ms/step - accuracy: 0.8935 - loss: 0.2989 - val_accuracy: 0.9021 - val_loss: 0.2915
Epoch 3/5
994/994 ━━━━━━━━━━━━━━━━━━━━ 31s 31ms/step - accuracy: 0.8996 - loss: 0.2904 - val_accuracy: 0.9012 - val_loss: 0.2870
Epoch 4/5
994/994 ━━━━━━━━━━━━━━━━━━━━ 25s 25ms/step - accuracy: 0.9022 - loss: 0.2832 - val_accuracy: 0.9072 - val_loss: 0.2873
Epoch 5/5
994/994 ━━━━━━━━━━━━━━━━━━━━ 27s 27ms/step - accuracy: 0.9059 - loss: 0.2762 - val_accuracy: 0.9044 - val_loss: 0.2862


<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
معیار ارزیابی
</font>
</h3>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    معیاری که برای ارزیابی عملکرد مدل انتخاب کرده‌ایم، <code>f1_score</code> نام دارد.
    <br>
    این معیار، سنجه ارزیابی کیفیت مدل شماست. به عبارت بهتر در سامانه داوری هم از همین معیار برای نمره‌دهی استفاده شده است.
    <br>
    این معیار برای هر مورد در ستون هدف به صورت جداگانه محاسبه شده و میانگین آن‌ها به عنوان امتیاز این مسئله در نظر گرفته می‌شود.
    <br>
    پیشنهاد می‌شود با توجه به این معیار، عملکرد مدل خود را بر روی مجموعه داده آموزش یا اعتبارسنجی ارزیابی کنید.
    
</font>
</p>

In [11]:
from sklearn.metrics import  f1_score, classification_report, confusion_matrix

val_generator.reset()
preds = model.predict(val_generator)

preds_for_f1 = np.where(preds >= 0.5, 1, 0)

y_true = val_generator.labels
y_true_for_f1 = np.where(y_true > 0.5, 1, 0)

f1 = f1_score(y_true_for_f1, preds_for_f1, average='macro')
print("Macro F1 Score on Validation set:", f1)

111/111 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step
Macro F1 Score on Validation set: 0.658570192680408


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
 پیش‌بینی برای داده تست و خروجی
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    پس از مهندسی ویژگی و مدلسازی، الگوریتمی دارید که می‌تواند شما را از متغیرهای مستقل به متغیرهای هدف برساند.
    <br>
    از این مدل برای پیش‌بینی نمونه‌های موجود در داده تست استفاده کنید و نتایج را در قالب جدول (<code>dataframe</code>) زیر آماده کنید.
</font>
</p>

<div dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|Male|مرد بودن|
|Young|جوان بودن|
|Middle_Aged|میانسال بودن|
|Senior|پیر بودن|
    
</font>
</div>



<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    اسم دیتافریم باید <i>submission</i> باشد؛ در غیر این صورت، سامانه داوری نمی‌تواند تلاش‌ شما را ارزیابی کند.
    <br>این دیتافریم تنها شامل ۴ ستون مربوط به ستون‌های هدف است و ۸۷۰۲ سطر دارد.
    <br>
    به ازای هر سطر موجود در دیتافریم <i>test</i> شما باید یک مقدار پیشبینی شده داشته باشید.
    <br>
    جدول زیر، ۵ سطر ابتدایی دیتافریم <code>submission</code> را نشان می‌دهد. البته در جواب شما، مقادیر ستون <i>TripReason</i> ممکن است متفاوت باشد.
</font>
</p>

<div style="text-align: center;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
| Male | Young | Middle_Aged | Senior |
|:----:|:-----:|:-----------:|:------:|
|  -1  |   1   |      -1     |    1   |
|  -1  |   1   |      -1     |    1   |
|  -1  |   1   |      -1     |    1   |
|  -1  |   1   |      -1     |    1   |
|  -1  |   1   |      -1     |    1   |

</font>
</div>



In [12]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_data,
    x_col='image_path',
    y_col=None,
    target_size=(128, 128),
    batch_size=32,
    class_mode=None, 
    shuffle=False
)

test_preds = model.predict(test_generator)

preds_binary = np.where(test_preds >= 0.5, 1, -1)

submission = pd.DataFrame(preds_binary, columns=target_cols)

submission.head()

Found 8702 validated image filenames.
271/272 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

E0000 00:00:1789813307.973526 3068893 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


272/272 ━━━━━━━━━━━━━━━━━━━━ 16s 51ms/step


,Male,Young,Middle_Aged,Senior
0,-1,1,-1,-1
1,-1,1,-1,-1
2,-1,1,-1,-1
3,-1,1,-1,-1
4,-1,1,-1,-1


<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) در غیر این صورت، در پایان مسابقه نمره شما به صفر تغییر خواهد کرد.
    <br>
    همچنین اگر از کولب برای اجرای این فایل نوت‌بوک استفاده می‌کنید، قبل از ارسال فایل <code>result.zip</code>، آخرین نسخه‌ی نوت‌بوک خود را دانلود کرده و داخل فایل ارسالی قرار دهید.
</font>

In [13]:
import zipfile
import joblib

if not os.path.exists(os.path.join(os.getcwd(), 'AzSorateshMalomeh.ipynb')):
    %notebook -e AzSorateshMalomeh.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)
            
submission.to_csv('submission.csv', index=False)
file_names = ['AzSorateshMalomeh.ipynb', 'submission.csv']
compress(file_names)

File Paths:
['AzSorateshMalomeh.ipynb', 'submission.csv']
